## Implementazione di un Bot Discord per la Classificazione CIFAR-10
In questa sezione integriamo il modello addestrato con un Bot Discord. 
Il bot riceverà un'immagine, la processerà e restituirà la classe predetta.

In [ ]:
# Installazione delle librerie necessarie
#!pip install discord.py tensorflow pillow nest-asyncio

import nest_asyncio
import io
import numpy as np
import tensorflow as tf
from PIL import Image
import discord
from discord.ext import commands
import os
from dotenv import load_dotenv 

# Patch necessaria per far girare il loop di Discord dentro Jupyter
nest_asyncio.apply()  #Ho incluso nest_asyncio perché i file Jupyter hanno spesso conflitti con i loop di Discord.

In [ ]:
import os
import discord
from discord.ext import commands
from dotenv import load_dotenv 

load_dotenv('TOKEN')
TOKEN = os.getenv('TOKEN')

##### 1. Caricamento del Modello e Definizione Classi
Carichiamo il file `.h5` generato durante la fase di training e definiamo le 10 etichette del dataset CIFAR-10.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

MODEL_PATH = r'C:\Users\Mary_Rosy\Desktop\gitdesktop\Progetto_Finale_Gruppo3\RESULT\cifar10_improved_model.keras'


try:
    # Aggiungiamo compile=False per ignorare l'ottimizzatore PyTorch incompatibile
    model = tf.keras.models.load_model(MODEL_PATH, compile=False)
    print("✅ Modello caricato correttamente (modalità inferenza)!")
    
    # Visualizziamo la struttura per essere sicuri che sia integro
    model.summary()
    
except Exception as e:
    print(f"❌ Errore nel caricamento: {e}")

class_names = ['aereo', 'automobile', 'uccello', 'gatto', 'cervo', 
               'cane', 'rana', 'cavallo', 'nave', 'camion']

# Preprocessing

## 2. Funzione di Pre-processing
Le immagini inviate su Discord possono avere qualsiasi dimensione. La funzione `prepare_image` si occupa di:
1. Convertire l'immagine in RGB.
2. Ridimensionarla a 32x32 pixel.
3. Normalizzare i valori dei pixel tra 0 e 1.

In [ ]:
def prepare_image(image_bytes):
    img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    img = img.resize((32, 32), Image.Resampling.LANCZOS)
    img_array = np.array(img).astype('float32') / 255.0
    return np.expand_dims(img_array, axis=0)

# Configurazione e Comandi Bot

### 3. Configurazione del Bot Discord
Definiamo il prefisso del comando (`!`) e la logica per gestire l'allegato. 
Il bot scaricherà l'immagine, userà il modello per la predizione e risponderà all'utente.

In [ ]:
# Configurazione permessi (Intents)
intents = discord.Intents.default()
intents.message_content = True 
bot = commands.Bot(command_prefix="!", intents=intents)

In [ ]:
import io
import time
import numpy as np
import discord
import matplotlib.pyplot as plt
from PIL import Image

@bot.command()
async def classifica(ctx):
    if not ctx.message.attachments:
        await ctx.send("⚠️ Allega una foto!")
        return

    attachment = ctx.message.attachments[0]
    loading_msg = await ctx.send("⌛ Inizializzazione scansione...")
    
    try:
        # 1. ELABORAZIONE IMMAGINE
        image_bytes = await attachment.read()
        pil_img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        
        # Versione 32x32 per il modello
        small_img = pil_img.resize((32, 32), Image.Resampling.LANCZOS)
        # Ingrandimento Nearest per la visualizzazione nitida
        display_img = small_img.resize((400, 400), Image.Resampling.NEAREST)
        
        # 2. INFERENZA E CALCOLO TEMPI
        processed_img = prepare_image(image_bytes)
        
        start_t = time.perf_counter()
        preds = model(processed_img, training=False)
        probs = preds[0].numpy()
        latenza_ms = (time.perf_counter() - start_t) * 1000
        
        # Dati per i risultati
        top3_idx = np.argsort(probs)[-3:][::-1]
        labels = [class_names[i].upper() for i in top3_idx]
        values = [probs[i] * 100 for i in top3_idx]

        # 3. CREAZIONE DASHBOARD GRAFICA (Occupa tutta la larghezza)
        # Creiamo un'unica immagine che affianca Foto e Grafico
        fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(12, 5))
        fig.patch.set_facecolor('#2f3136') # Colore simile a Discord
        
        # Parte sinistra: Immagine
        ax_img.imshow(display_img)
        ax_img.axis('off')
        ax_img.set_title("VISTA MODELLO (32x32)", color='white', fontsize=12, fontweight='bold')

        # Parte destra: Grafico a barre
        y_pos = np.arange(len(labels))
        bars = ax_bar.barh(y_pos, values[::-1], color='#3498db', height=0.6)
        ax_bar.set_yticks(y_pos)
        ax_bar.set_yticklabels(labels[::-1], color='white', fontsize=11, fontweight='bold')
        ax_bar.set_xlim(0, 115) # Spazio extra per le etichette di testo
        
        # Estetica grafico
        ax_bar.patch.set_facecolor('none')
        ax_bar.spines['top'].set_visible(False)
        ax_bar.spines['right'].set_visible(False)
        ax_bar.spines['bottom'].set_color('white')
        ax_bar.spines['left'].set_color('white')
        ax_bar.tick_params(axis='x', colors='white')
        
        # Aggiunta percentuali accanto alle barre
        for i, v in enumerate(values[::-1]):
            ax_bar.text(v + 2, i, f"{v:.1f}%", color='white', va='center', fontweight='bold')

        plt.tight_layout()
        
        # Salvataggio Dashboard
        buf = io.BytesIO()
        plt.savefig(buf, format='png', facecolor=fig.get_facecolor())
        buf.seek(0)
        file_dash = discord.File(fp=buf, filename="dashboard.png")
        plt.close()

        # 4. COSTRUZIONE EMBED BILANCIATO
        embed = discord.Embed(
            title="🔬 Report Analisi MaGMI", 
            description=f"Analisi completata per l'utente **{ctx.author.name}**",
            color=0x3498db
        )
        
        # Questa immagine ora riempirà tutta la zona centrale, eliminando i vuoti
        embed.set_image(url="attachment://dashboard.png")
        
        # Usiamo inline=True per riempire la riga orizzontalmente
        embed.add_field(name="🎯 Verdetto", value=f"**{labels[0]}**", inline=True)
        embed.add_field(name="📈 Confidenza", value=f"{values[0]:.2f}%", inline=True)
        embed.add_field(name="⏱️ Latenza", value=f"{latenza_ms:.2f} ms", inline=True)
        
        # Footer istituzionale
        embed.set_footer(text="UniBa Aldo Moro | Laboratorio MaGMI | Fisica Digitale")

        await loading_msg.delete()
        await ctx.send(file=file_dash, embed=embed)

    except Exception as e:
        await ctx.send(f"❌ Errore critico: {e}")

In [ ]:
'''Il bot verifica immediatamente se l'utente ha allegato un file. 
Se il messaggio è vuoto, interrompe l'esecuzione e invia un avviso.'''

'''@bot.command()
async def classifica(ctx):
    if not ctx.message.attachments:
        await ctx.send("⚠️ Allega una foto!")
        return

    attachment = ctx.message.attachments[0]
    
    # 1. Messaggio iniziale (Stato t=0)
    loading_msg = await ctx.send("⌛ Inizializzazione scansione...")
    
    try:
        # 2. Simulazione progresso visivo (Fase di pre-processing)
        # Questo mima la latenza di caricamento del segnale
        await loading_msg.edit(content="📡 `███░░░░░░░` Acquisizione segnale...")
        
        image_bytes = await attachment.read()
        processed_img = prepare_image(image_bytes)
        
        # 3. Simulazione progresso (Fase di inferenza)
        await loading_msg.edit(content="🧠 `███████░░░` Analisi neurale in corso...")
        
        # Esecuzione effettiva del modello
        preds = model(processed_img, training=False)
        #preds contiene i punteggi (spesso logit) per le 10 classi di CIFAR-10.
        #np.argmax(probs) individua l'indice con il valore di probabilità più alto.
        probs = preds[0].numpy()
        index = np.argmax(probs)
        confidenza = probs[index] * 100
        
        # 4. Completamento scansione
        await loading_msg.edit(content="✅ `██████████` Scansione completata!")
        
        # Creazione dell'Embed finale
        embed = discord.Embed(title="🔍 Analisi CIFAR-10", color=0x3498db)
        embed.set_thumbnail(url=attachment.url)
        embed.add_field(name="Oggetto Identificato", value=f"✨ **{class_names[index].upper()}**", inline=False)
        embed.add_field(name="Grado di Sicurezza", value=f"📈 {confidenza:.2f}%", inline=True)
        
        embed.set_footer(text=f"AI Model: MaGMI image recognizer bot | Crediti: MaGMI")

        # Rimuoviamo il messaggio di caricamento e inviamo il risultato
        await loading_msg.delete()
        await ctx.send(embed=embed)

    except Exception as e:
        await loading_msg.edit(content=f"❌ Errore durante la scansione: {e}")'''

### Gestione Errori

In [ ]:
# 1. Configurazione Stato Personalizzato
@bot.event
async def on_ready():
    # Imposta lo stato: "In ascolto di !info"
    activity = discord.Activity(type=discord.ActivityType.listening, name="!info")
    await bot.change_presence(status=discord.Status.online, activity=activity)
    print(f'✅ Bot online come: {bot.user}')

# 2. Gestione Errori Globale
@bot.event
async def on_command_error(ctx, error):
    if isinstance(error, commands.CommandNotFound):
        await ctx.send("❓ Comando non riconosciuto. Scrivi `!info` per vedere cosa posso fare.")
    elif isinstance(error, commands.MissingRequiredArgument):
        await ctx.send("⚠️ Mancano degli argomenti nel comando.")
    else:
        print(f"Errore non gestito: {error}")

### Comando Informativo
Aggiungiamo un comando per spiegare agli utenti quali sono le 10 categorie che il modello può riconoscere (dataset CIFAR-10) e come utilizzare il bot.

In [ ]:
@bot.command()
async def info(ctx):
    # Creiamo un messaggio formattato in modo elegante (Embed)
    descrizione = (
        "Ciao! Sono il Bot del progetto di Classificazione Immagini.\n"
        "Utilizzo una rete neurale (CNN) addestrata sul dataset **CIFAR-10**.\n\n"
        "**Cosa posso fare?**\n"
        "Se mi invii una foto e scrivi `!classifica`, proverò a capire cosa rappresenta.\n\n"
        "**Le mie 10 categorie sono:**\n"
        f"✈️ {class_names[0]}, 🚗 {class_names[1]}, 🐦 {class_names[2]}, 🐱 {class_names[3]}, 🦌 {class_names[4]},\n"
        f"🐶 {class_names[5]}, 🐸 {class_names[6]}, 🐴 {class_names[7]}, 🚢 {class_names[8]}, 🚛 {class_names[9]}.\n\n"
        "**Istruzioni:**\n"
        "Carica una foto e scrivi `!classifica` nel campo del commento!"
    )
    
    await ctx.send(descrizione)

# Esecuzione
Inserisci il tuo Token e avvia la cella. Il bot rimarrà attivo finché la cella è in esecuzione.

In [ ]:
bot.run(TOKEN)